# 批量处理 MTSD 图片

使用 `slicer.py` 中的 `process_image` 方法批量处理所有图片，并保存到目标路径。

## 数据路径
- 输入：`/Users/weixianfu/Documents/Datas/mtsd` (train_full, train_partial, val)
- 输出：`/Users/weixianfu/Documents/Datas/mtsd-resized` (保持相同文件夹结构)

In [9]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

sys.path.append(str(Path.cwd().parent))
from src.io import process_image

In [10]:
INPUT_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd")
OUTPUT_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd-resized")
DATASET_TYPES = ["train_full", "train_partial", "val"]
MAX_WORKERS = 8

In [11]:
def get_image_names(dataset_type: str) -> list:
    images_dir = INPUT_ROOT / dataset_type / "images"
    image_names = []
    for img_path in images_dir.glob("*.jpg"):
        image_names.append(img_path.stem)
    return image_names

def process_single_image(name: str, dataset_type: str):
    input_path = INPUT_ROOT / dataset_type
    output_path = OUTPUT_ROOT / dataset_type
    process_image(name=name, input_path=input_path, output_path=output_path)
    return name

In [12]:
for dataset_type in DATASET_TYPES:
    print(f"\n处理 {dataset_type}...")
    image_names = get_image_names(dataset_type)
    print(f"找到 {len(image_names)} 张图片")
    
    if len(image_names) == 0:
        print(f"跳过 {dataset_type}，没有找到图片")
        continue
    
    output_path = OUTPUT_ROOT / dataset_type
    (output_path / "images").mkdir(parents=True, exist_ok=True)
    (output_path / "labels").mkdir(parents=True, exist_ok=True)
    print(f"输出目录已准备: {output_path}")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_single_image, name, dataset_type): name 
            for name in image_names
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"处理 {dataset_type}"):
            name = futures[future]
            future.result()
    
    print(f"{dataset_type} 处理完成")


处理 train_full...
找到 36589 张图片


处理 train_full:   0%|          | 0/36589 [00:00<?, ?it/s]
Premature end of JPEG file


KeyboardInterrupt: 